# 📖 Notebook 4: Advanced Gateway Patterns

You've seen the core gateway responsibilities (routing, rate limiting, auth, transformation). This notebook covers the **advanced patterns** that turn a basic gateway into a production-grade one:

| Pattern | What it solves |
|---------|----------------|
| **Circuit-breaker-style failover** | Stops sending traffic to a backend that keeps failing |
| **Retry amplification** | Explains why retries at every layer multiply load instead of adding it |
| **Timeout budgets** | Keeps a slow dependency from becoming a dead client connection |
| **Request aggregation (BFF)** | Lets clients make one call instead of many |
| **Canary / weighted routing** | Ships a new version to a small % of users |
| **CORS** | Lets browsers from other origins call your API safely |
| **Observability (logging/tracing)** | Makes it possible to debug what went wrong, in which service |
| **SSL/TLS termination** | Terminates HTTPS at the edge so backends stay simple |

We'll follow the same 🚫 BAD → ✅ BETTER → 🏆 BEST structure wherever it fits.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why you need failover and what nginx can (and cannot) do for you
- The difference between true circuit breakers and passive failover
- Why nested retries multiply load, and how to bound them
- Why a timeout budget has to shrink as you go deeper into the call graph
- When to aggregate requests at the gateway vs. in a dedicated BFF, and how the aggregate degrades when one upstream is down
- How canary / weighted routing lets you safely ship a new version to a small % of users
- How CORS preflight requests work and how the gateway handles them
- How `X-Request-ID` enables distributed tracing across services
- Why almost every production gateway terminates TLS


## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd 05-microservices/api-gateway
docker compose up -d --build
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".


In [ ]:
import requests
import json
import time

GATEWAY = "http://localhost:8080"

def show(response):
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text[:300])

try:
    r = requests.get(f"{GATEWAY}/health", timeout=3)
    print(f"✅ API Gateway: {r.json()['status']}")
except Exception as e:
    print(f"❌ API Gateway not running: {e}")
    print("   Run: cd 05-microservices/api-gateway && docker compose up -d --build")


---

## 1️⃣ Circuit-Breaker-Style Failover

### The problem

One of your backend instances starts failing (network glitch, OOM, bad deploy). If the gateway keeps sending it traffic, every request to that instance fails. Worse: the gateway itself slows down while it waits for timeouts. This is called a **cascading failure**.

### 🚫 BAD: No failover

With a naïve load balancer, 50% of your traffic goes to the dead instance → 50% of users see errors.

### ✅ BETTER: Passive failover (what nginx gives us)

Our `nginx.conf` declares each upstream with:

```nginx
upstream user_backend {
    server user-service-1:5000 max_fails=3 fail_timeout=30s;
    server user-service-2:5000 max_fails=3 fail_timeout=30s;
}

proxy_next_upstream         error timeout http_502 http_503 http_504;
proxy_next_upstream_tries   2;    # at most 2 upstreams attempted
proxy_next_upstream_timeout 5s;   # ...and at most 5s spent across all of them
proxy_connect_timeout       2s;
proxy_read_timeout          5s;
```

Behavior:
- If a backend fails/times out, nginx retries the **next** healthy instance automatically
- After 3 failures in 30s, nginx marks the instance "down" and stops sending it traffic
- After 30s nginx tries again (very coarse "recovery" check)

### 🏆 BEST: True circuit breaker (what you'd use in production)

A real circuit breaker has three states:

```
     closed  ───failures exceed threshold──▶   open
       ▲                                         │
       │                                         │ wait timeout
       │                                         ▼
   success ◀────probe succeeds────   half-open
```

- **closed** — normal traffic flow
- **open** — all requests fail fast (no backend call made at all)
- **half-open** — let a few probe requests through; if they succeed, close; if they fail, re-open

Production tools that implement this: **Envoy**, **Istio**, **Linkerd**, **Resilience4j** (Java), **Polly** (.NET), **pybreaker** (Python). nginx OSS does not.

Let's simulate the pattern in Python so you understand the state machine:


In [ ]:
# A minimal circuit breaker — the pattern your service mesh implements for you.


class CircuitOpen(RuntimeError):
    """Raised when the breaker refuses to even attempt the call.

    Worth its own type: the whole value of a breaker is that this failure is
    FREE. No connection, no timeout, no load on a backend that is already
    struggling. A caller that can't tell this apart from a real backend error
    can't do the sensible thing (serve a cached/degraded response instantly).
    """


class CircuitBreaker:
    CLOSED, OPEN, HALF_OPEN = "closed", "open", "half_open"

    def __init__(self, fail_threshold=3, recovery_seconds=5, clock=time.monotonic):
        self.state = self.CLOSED
        self.failures = 0
        self.opened_at = 0.0
        self.fail_threshold = fail_threshold
        self.recovery_seconds = recovery_seconds
        # The clock is injected so we can drive the state machine with a fake
        # one below. Timing-dependent demos that "work on my laptop" are how
        # you end up with a lab that quietly stops teaching its own lesson.
        self.clock = clock
        self.transitions = []   # every state change, in order

    def _to(self, state):
        if state != self.state:
            self.state = state
            self.transitions.append(state)

    def call(self, fn, *args, **kwargs):
        # If OPEN, short-circuit until the recovery window passes
        if self.state == self.OPEN:
            if self.clock() - self.opened_at >= self.recovery_seconds:
                self._to(self.HALF_OPEN)   # let a single probe request through
                print("   ↩️  HALF_OPEN: sending a probe request...")
            else:
                raise CircuitOpen("circuit OPEN — failing fast, backend not called")

        try:
            result = fn(*args, **kwargs)
        except Exception as exc:
            self.failures += 1
            # A failed probe re-opens immediately — one bad probe is enough,
            # we do not spend another fail_threshold requests finding out.
            if self.state == self.HALF_OPEN or self.failures >= self.fail_threshold:
                self._to(self.OPEN)
                self.opened_at = self.clock()
                print(f"   🔥 circuit OPENED after failure: {exc}")
            raise
        else:
            if self.state == self.HALF_OPEN:
                print("   ✅ probe succeeded — circuit CLOSED")
            self._to(self.CLOSED)
            self.failures = 0
            return result


class VirtualClock:
    """A fake monotonic clock. One tick per request, no sleeping, no flakiness."""

    def __init__(self, step=1.0):
        self.now = 0.0
        self.step = step

    def __call__(self):
        return self.now

    def tick(self):
        self.now += self.step


# Simulate: a backend that fails its first 5 calls, then recovers
attempts = {"n": 0}


def flaky_backend():
    attempts["n"] += 1
    if attempts["n"] <= 5:
        raise RuntimeError("backend down")
    return "ok"


clock = VirtualClock(step=1.0)
cb = CircuitBreaker(fail_threshold=3, recovery_seconds=3, clock=clock)

short_circuited = 0
for i in range(15):
    try:
        out = cb.call(flaky_backend)
        print(f"Request {i+1:>2}: t={clock.now:>4.0f}s  state={cb.state:<10} result={out}")
    except CircuitOpen as exc:
        short_circuited += 1
        print(f"Request {i+1:>2}: t={clock.now:>4.0f}s  state={cb.state:<10} error={exc}")
    except Exception as exc:
        print(f"Request {i+1:>2}: t={clock.now:>4.0f}s  state={cb.state:<10} error={exc}")
    clock.tick()

print()
print("State transitions:", " → ".join(["closed"] + cb.transitions))
print(f"Client requests:            15")
print(f"Calls that reached backend: {attempts['n']}")
print(f"Calls short-circuited:      {short_circuited}  ← free failures, zero backend load")

# The point of the breaker is BOTH halves of this:
#   (a) it walks the full closed → open → half-open → closed cycle, and
#   (b) it sheds load off a backend that is already on fire.
assert cb.transitions == ["open", "half_open", "open", "half_open", "open", "half_open", "closed"], (
    f"expected the breaker to probe and re-open twice before recovering, got {cb.transitions}")
assert cb.state == "closed", f"breaker should have recovered, ended in {cb.state}"
assert attempts["n"] == 9 and short_circuited == 6, (
    f"breaker should have spared the backend 6 of 15 requests, "
    f"got {attempts['n']} backend calls / {short_circuited} short-circuits")

### What nginx gives us vs. what a service mesh gives us

| Capability | nginx OSS (`max_fails`/`proxy_next_upstream`) | Envoy / Istio |
|-----------|:---:|:---:|
| Retry next instance on error | ✅ | ✅ |
| Eject failing instance temporarily | ✅ (coarse) | ✅ (fine-grained) |
| Open / Half-Open / Closed state machine | ❌ | ✅ |
| Fail-fast when circuit is open | ❌ | ✅ |
| Health checks (active) | ⚠️ nginx Plus only | ✅ |
| Metrics for breaker state | ❌ | ✅ |

**Takeaway:** for a small system, `max_fails` + timeouts is often enough. When you grow to many services and need production-grade resilience, move to a service mesh or a dedicated client-side breaker library.


---

## 1️⃣ b. Retry amplification — the reason retries are dangerous

Every layer in the diagram wants to be helpful. The mobile SDK retries. The
gateway retries the next upstream. The service retries its own dependency. Each
of those is individually reasonable, and **they multiply**.

```
client retries 3x ──▶ gateway retries 2x ──▶ service retries 3x ──▶ backend
      1 user request                                          3 x 2 x 3 = 18 calls
```

The multiplier is the **product** of the per-layer retry counts, not the sum.
And the cruelty of it is the timing: amplification is smallest when everything
is healthy (almost nothing retries) and largest exactly when the backend is
already failing. A backend at 90% capacity gets a small blip, retries kick in,
offered load jumps several-fold, more requests fail, which triggers more
retries. That feedback loop is a **metastable failure**: the system stays down
after the original trigger is gone, because the retries are now the load.

Our `nginx.conf` deliberately keeps `proxy_next_upstream_tries 2` and
`user_service.py` does not retry its call to order-service at all, so the lab's
own worst case is 2x, not 18x. Let's measure what the difference looks like.

In [ ]:
# Measure retry amplification: backend calls per user request, as failures rise.

import math
import random


def simulate_retries(p_fail, tries_per_layer, n_requests=20000, seed=42):
    """Return (backend calls per user request, fraction of user requests that succeeded).

    tries_per_layer is outermost-first, e.g. [3, 2, 3] = client, gateway, service.
    A "try" is one attempt; tries=1 means "no retry".
    """
    rng = random.Random(seed)          # seeded: same numbers on every machine
    calls = 0

    def attempt(depth):
        nonlocal calls
        if depth == len(tries_per_layer):
            calls += 1                  # this is the backend actually doing work
            return rng.random() >= p_fail
        for _ in range(tries_per_layer[depth]):
            if attempt(depth + 1):
                return True
        return False

    ok = sum(attempt(0) for _ in range(n_requests))
    return calls / n_requests, ok / n_requests


NESTED = [3, 2, 3]   # client x gateway x service  -> worst case 18
SINGLE = [1, 2, 1]   # only the gateway retries    -> worst case 2 (this lab)

print("Backend calls per user request as the backend degrades")
print("=" * 68)
print(f"{'p(fail)':>8} | {'nested 3x2x3':>14} {'success':>8} | {'gateway-only 2x':>16} {'success':>8}")
print("-" * 68)

nested_mults, single_mults = [], []
for p in [0.0, 0.05, 0.2, 0.5, 0.8, 1.0]:
    n_mult, n_ok = simulate_retries(p, NESTED)
    s_mult, s_ok = simulate_retries(p, SINGLE)
    nested_mults.append(n_mult)
    single_mults.append(s_mult)
    print(f"{p:>8.2f} | {n_mult:>14.2f} {n_ok:>7.1%} | {s_mult:>16.2f} {s_ok:>7.1%}")

worst_nested = math.prod(NESTED)
print()
print(f"💥 At p=1.00 the nested config sends {nested_mults[-1]:.0f} calls per user request")
print(f"   — exactly the product {' x '.join(map(str, NESTED))} = {worst_nested}.")
print("   The backend that could not serve 1 request/user is now asked for 18.")
print()
print("👀 Read the success columns too -- retries are not simply bad. At p=0.80")
print("   the nested config still serves 98% of users while the gateway-only")
print("   config serves 36%. Retries BUY availability; the price is a load")
print("   multiplier that peaks exactly when you can least afford it. The job")
print("   is to buy that availability once, at one layer, with a budget.")
print()
print("🛠️  How production systems keep retries safe:")
print("   • Retry at ONE layer (usually the outermost that can still act on it)")
print("   • Retry BUDGETS: cap retries at e.g. 10% of total requests, not per-call")
print("   • Exponential backoff + jitter, so retries spread out instead of syncing")
print("   • Don't retry when a circuit breaker is open, or on non-retryable errors")
print("   • Only retry IDEMPOTENT operations — a retried POST can double-charge")

# The lesson is the product rule and its timing. Both must hold.
assert nested_mults[-1] == worst_nested, (
    f"a fully-failing backend must see the product of the retry counts "
    f"({worst_nested}), got {nested_mults[-1]}")
assert nested_mults == sorted(nested_mults), (
    "amplification must grow as the backend gets sicker -- that is the trap")
assert all(n >= s for n, s in zip(nested_mults, single_mults)), (
    "nested retries should never send fewer calls than single-layer retries")
assert nested_mults[-1] / single_mults[-1] == worst_nested / math.prod(SINGLE), (
    "the gap between the two configs should be the ratio of their products")

---

## 2️⃣ Request Aggregation (Backend-For-Frontend pattern)

### The problem

A mobile "profile screen" needs: the user's details **and** their orders. Those live in two services.

### 🚫 BAD: The client makes N calls

```
Mobile app                Network
────────                  ───────
GET /api/users/1    ──────▶   (round-trip #1, ~200ms on mobile)
GET /api/orders?user_id=1 ─▶  (round-trip #2, ~200ms on mobile)
```

Two round-trips means twice the latency — painful on mobile networks — and forces the client to handle partial failures.

### 🏆 BEST: The gateway (or BFF) aggregates

```
Mobile app        Gateway / BFF          Services
────────          ─────────────          ────────
GET /api/profile/1 ─▶ fetch user ──────▶ user-service
                      fetch orders ────▶ order-service
                   ◀── combined JSON
```

One round-trip, one combined payload, and partial failures are handled server-side.

Let's hit the aggregated endpoint:


In [ ]:
time.sleep(1)  # avoid rate limiting from earlier notebooks

print("🏆 Aggregated profile endpoint")
print("=" * 55)
print("ONE call to the gateway returns user + orders:")
print()

r = requests.get(f"{GATEWAY}/api/profile/1")
show(r)


In [ ]:
# Compare: the client-side approach (what you would have WITHOUT aggregation)
# We sleep briefly between calls to stay under the gateway's 5 req/sec rate limit.

def client_side_composition(user_id):
    r1 = requests.get(f"{GATEWAY}/api/users/{user_id}")
    time.sleep(0.25)
    r2 = requests.get(f"{GATEWAY}/api/orders", params={"user_id": user_id})
    return {"user": r1.json(), "orders": r2.json().get("orders", [])}

def gateway_composition(user_id):
    return requests.get(f"{GATEWAY}/api/profile/{user_id}").json()

N = 5  # small so we don't trip the rate limiter

t0 = time.time()
for _ in range(N):
    client_side_composition("1")
    time.sleep(0.25)
client_total_ms = (time.time() - t0) / N * 1000

time.sleep(1.5)

t0 = time.time()
for _ in range(N):
    gateway_composition("1")
    time.sleep(0.25)
gateway_total_ms = (time.time() - t0) / N * 1000

# Subtract the throttle sleeps so the numbers reflect real work, not our pause.
THROTTLE_MS = 250
client_ms  = client_total_ms  - THROTTLE_MS * 2  # 2 sleeps per iteration
gateway_ms = gateway_total_ms - THROTTLE_MS * 1  # 1 sleep per iteration

print(f"Client-side (2 calls/req):  avg {client_ms:.1f} ms per profile")
print(f"Gateway-side (1 call/req):  avg {gateway_ms:.1f} ms per profile")
print(f"Saved:                      {client_ms - gateway_ms:.1f} ms  "
      f"({(1 - gateway_ms / client_ms) * 100:.0f}% less)")
print()
print("What you are really measuring: two client→gateway round trips versus one.")
print("The second hop in the gateway case (user-service → order-service) happens")
print("on the container network and is nearly free, which is the whole point --")
print("you moved a slow, far hop into a fast, near one.")
print()
print("⚖️  Two honest caveats before you quote this number in an interview:")
print("   1. A real client could fire its two calls IN PARALLEL, and then the")
print("      latency win mostly disappears. The durable wins are fewer")
print("      connections, less battery/radio use, a smaller payload, and one")
print("      place to handle partial failure — not raw wall-clock latency.")
print("   2. Aggregation moves the fan-out, it does not remove it. The BFF now")
print("      owns the timeout budget, the partial-failure story, and the blast")
print("      radius when one dependency is slow. That is the next two cells.")

# The aggregated endpoint must actually be the cheaper shape. If this flips,
# the aggregation route is broken (or the comparison stopped being fair).
assert gateway_ms < client_ms, (
    f"one round-trip should beat two: gateway {gateway_ms:.1f} ms vs "
    f"client-side {client_ms:.1f} ms")
assert gateway_ms > 0 and client_ms > 0, (
    f"throttle subtraction produced a negative time (client={client_ms:.1f}, "
    f"gateway={gateway_ms:.1f}) -- the sleeps are dominating the measurement")


### Where should composition live?

| Option | Pros | Cons |
|--------|------|------|
| **In the gateway itself** (nginx + Lua / Kong) | One hop, centralized | Gateway becomes smart/stateful |
| **Dedicated BFF service** (one per client type) | Tailored per client (iOS vs Web) | Another service to own |
| **GraphQL gateway** (Apollo, Hasura) | Clients request exactly the fields they need | Schema/tooling complexity |
| **Inside a backend service** (what this lab does) | Simplest | Violates single-responsibility |

For this lab, the aggregation lives in `user_service.py` to keep the container count low. In a real system, prefer a **dedicated BFF** or a **GraphQL gateway** — especially if different client types (iOS/Android/Web) need different payloads.

### Partial-failure behavior

Notice the aggregated endpoint uses a short (1.5s) timeout when calling order-service. If that call fails, the endpoint still returns user data with `orders_unavailable: true`. This **graceful degradation** is a must-have pattern in any aggregation layer — one slow dependency should never take down the whole response.


### Prove the partial-failure path, don't just claim it

The paragraph above says the aggregated endpoint degrades gracefully. That's the
kind of claim that is true right up until someone edits the `except` clause.

We can't take order-service down here — other people are using this stack — so
instead we run **the same composition logic** against two broken dependencies we
create locally:

1. a stub order service that answers, but **too slowly** (exercises the timeout)
2. a port with **nothing listening** (exercises connection refused)

Watch two things: the whole call stays inside the timeout budget, and
`order_count` comes back as `null` rather than `0`. That second one matters more
than it looks. `0` is a *claim* — "this user has no orders" — and we did not
verify it. Returning `0` for "I don't know" is how a UI ends up cheerfully
telling a customer their order history is empty during an outage.

In [ ]:
# Reproduce both failure modes of the aggregation call, locally.

import http.server
import json as _json
import socketserver
import threading
import urllib.error
import urllib.request

ORDER_TIMEOUT = 1.5      # matches _fetch_orders_for_user in services/user_service.py
STUB_DELAY = 5.0         # stub answers far too late


class _SlowOrders(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        time.sleep(STUB_DELAY)
        body = _json.dumps({"orders": []}).encode()
        try:
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)
        except OSError:
            pass          # the client gave up long ago -- that is the point

    def log_message(self, *args):
        pass


class _QuietServer(socketserver.ThreadingTCPServer):
    daemon_threads = True
    allow_reuse_address = True

    def handle_error(self, *args):
        pass


stub = _QuietServer(("127.0.0.1", 0), _SlowOrders)
threading.Thread(target=stub.serve_forever, daemon=True).start()
SLOW_URL = f"http://127.0.0.1:{stub.server_address[1]}"
DEAD_URL = "http://127.0.0.1:1"          # nothing listens on port 1


def fetch_orders(base_url, user_id, timeout_seconds=ORDER_TIMEOUT):
    """Same body as services/user_service.py::_fetch_orders_for_user."""
    url = f"{base_url}/orders?user_id={user_id}"
    try:
        with urllib.request.urlopen(url, timeout=timeout_seconds) as resp:
            return _json.loads(resp.read().decode("utf-8")).get("orders", []), None
    except (urllib.error.URLError, TimeoutError, ValueError) as exc:
        return [], str(exc)


def compose_profile(base_url, user_id="1"):
    """Same composition as services/user_service.py::get_user_profile."""
    user = {"id": user_id, "name": "Alice Johnson", "email": "alice@example.com"}
    orders, error = fetch_orders(base_url, user_id)
    profile = {
        "user": user,
        "orders": orders,
        "order_count": len(orders) if error is None else None,
        "served_by": "user-service-local-sim",
    }
    if error is not None:
        profile["orders_unavailable"] = True
        profile["orders_error"] = error
    return profile


print("🩺 Aggregation under partial failure")
print("=" * 60)

for label, url in [("order-service too SLOW", SLOW_URL), ("order-service DEAD", DEAD_URL)]:
    t0 = time.time()
    profile = compose_profile(url)
    elapsed = time.time() - t0

    print(f"\n{label}  (took {elapsed:.2f}s, budget {ORDER_TIMEOUT}s)")
    print(f"  user returned:      {profile['user']['name']}  ← the page still renders")
    print(f"  order_count:        {profile['order_count']}  ← null, NOT 0")
    print(f"  orders_unavailable: {profile.get('orders_unavailable')}")
    print(f"  orders_error:       {profile.get('orders_error')}")

    # 1. The dependency's slowness must not become OUR slowness.
    assert elapsed < ORDER_TIMEOUT + 1.0, (
        f"{label}: composition took {elapsed:.2f}s, the {ORDER_TIMEOUT}s timeout did not bound it")
    # 2. Degrade, don't fail: the half we could fetch still comes back.
    assert profile["user"]["name"] == "Alice Johnson", f"{label}: lost the user data too"
    # 3. Never report an unverified fact as a number.
    assert profile["order_count"] is None, (
        f"{label}: order_count={profile['order_count']} claims we know the order "
        f"count, but the call failed -- it must be null")
    assert profile["orders_unavailable"] is True, f"{label}: partial failure was not signalled"

stub.shutdown()
print()
print("✅ Both failure modes degrade instead of failing, inside the timeout budget.")
print("   The client can now render the profile page and show 'orders unavailable'")
print("   in the orders panel — instead of a blank screen or a wrong '0 orders'.")

### Timeout budgets have to shrink as you go deeper

The aggregation call above is the first place in this lab with **two nested
hops**, so it's the first place a timeout budget can be wrong.

The rule: every level of the call graph must allow **less** time than the level
above it. If an inner hop is given the same deadline as an outer one, the outer
hop times out first — and now nobody benefits: the client sees a dead
connection, the inner work keeps running (burning capacity for a response no one
will read), and any retry the outer layer makes is *stacked on top* of work that
is still in flight. That's retry amplification again, arriving through the back
door.

```
client              deadline 10s
  └─ gateway        proxy_read_timeout  5s     ← 5s of headroom for the client
       └─ user-service → order-service  1.5s   ← 3.5s of headroom for the gateway
```

The headroom at each level is what pays for the *degraded* answer. Because
user-service gives up at 1.5s, the gateway still has 3.5s in hand to return a
profile with `orders_unavailable: true` — a useful response — instead of the
client's connection dying at 10s with nothing at all.

Nothing enforces this automatically. It's a set of numbers in two different
files that a well-meaning edit can silently break, so let's assert on it.

In [ ]:
# Read the real timeout numbers out of the config files and check the budget shrinks.

import re
from pathlib import Path

LAB = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "gateway" / "nginx.conf").exists())

nginx_conf = (LAB / "gateway" / "nginx.conf").read_text()
user_svc = (LAB / "services" / "user_service.py").read_text()

gateway_read = float(re.search(r"proxy_read_timeout\s+(\d+(?:\.\d+)?)s", nginx_conf).group(1))
gateway_connect = float(re.search(r"proxy_connect_timeout\s+(\d+(?:\.\d+)?)s", nginx_conf).group(1))
next_tries = int(re.search(r"proxy_next_upstream_tries\s+(\d+)", nginx_conf).group(1))
next_cap = float(re.search(r"proxy_next_upstream_timeout\s+(\d+(?:\.\d+)?)s", nginx_conf).group(1))
service_call = float(re.search(r"timeout_seconds:\s*float\s*=\s*([\d.]+)", user_svc).group(1))

CLIENT_DEADLINE = 10.0   # what a mobile client would typically allow

budget = [
    ("client",                       CLIENT_DEADLINE, "app-side deadline"),
    ("gateway → service",            gateway_read,    "nginx proxy_read_timeout"),
    ("service → order-service",      service_call,    "urlopen(timeout=...) in user_service.py"),
]

print("⏱️  Timeout budget, outermost first")
print("=" * 66)
prev = None
for name, seconds, where in budget:
    headroom = "" if prev is None else f"  (leaves {prev - seconds:.1f}s of headroom above)"
    print(f"  {name:<26} {seconds:>5.1f}s   {where}{headroom}")
    prev = seconds

print()
print(f"  gateway connect timeout:    {gateway_connect:>4.1f}s  (must fit inside the read timeout)")
print()
print("⚠️  Retries break naive budget arithmetic. proxy_next_upstream_tries="
      f"{next_tries} means")
print(f"   the gateway may attempt {next_tries} upstreams, so a naive worst case is")
print(f"   {next_tries} x {gateway_read:.0f}s = {next_tries * gateway_read:.0f}s — which would blow the "
      f"client's {CLIENT_DEADLINE:.0f}s deadline.")
print(f"   That is exactly why proxy_next_upstream_timeout {next_cap:.0f}s exists: it caps the")
print("   TOTAL time spent across all attempts. A retry count without a total")
print("   cap is an unbounded latency budget wearing a number.")

# The budget must strictly shrink. If someone bumps an inner timeout to
# "make flaky calls work", this fails and says why.
seconds = [s for _, s, _ in budget]
assert seconds == sorted(seconds, reverse=True) and len(set(seconds)) == len(seconds), (
    f"timeout budget must strictly shrink with depth, got {seconds}")
assert gateway_connect < gateway_read, (
    f"connect timeout ({gateway_connect}s) must be smaller than the read timeout ({gateway_read}s)")
# Retries must be bounded by a TOTAL cap, not just a count.
assert next_cap <= gateway_read, (
    f"proxy_next_upstream_timeout ({next_cap}s) must cap retries at or below one "
    f"upstream read timeout ({gateway_read}s), otherwise retries extend the budget")
assert next_cap < CLIENT_DEADLINE, (
    f"the gateway's capped retry window ({next_cap}s) must still fit inside the "
    f"client's {CLIENT_DEADLINE}s deadline")

---

## 3️⃣ Canary / Weighted Routing

### The problem

You want to ship a new version of a service but you don't want to blast 100% of real users with it on day 1. If there's a bug, everyone is impacted at once.

### 🏆 BEST: Send a small slice of traffic to the new version

A **canary release** deploys the new version alongside the old one and sends only a small percentage (e.g. 5–10%) of traffic to it. You watch dashboards: if error rate or latency stays flat, you **ramp up** the canary weight (10% → 50% → 100%). If anything looks bad, you drop it back to 0% with a single config reload — no redeploy, no panic.

```
             ┌──────────────┐
             │  API Gateway │
             └──────┬───────┘
                    │
          90% │    10% │ (canary weight)
              ▼        ▼
      ┌──────────┐  ┌──────────┐
      │  stable  │  │  canary  │   ← new version, small blast radius
      │  v1.4.0  │  │  v1.5.0  │
      └──────────┘  └──────────┘
```

### How nginx does it — weighted upstreams

Our `nginx.conf` declares:

```nginx
upstream user_canary_backend {
    server user-service-1:5000 weight=9;   # "stable" — 90% of traffic
    server user-service-2:5000 weight=1;   # "canary" — 10% of traffic
}

location /api/canary/users {
    proxy_pass http://user_canary_backend/users;
}
```

In this lab, `user-service-1` and `user-service-2` are the **same code** — we just pretend one is the new version. In production these would be two different image tags or git commits.

Let's send 100 requests through `/api/canary/users` and see the split:


In [ ]:
time.sleep(1)  # avoid carry-over rate limiting

print("\U0001f424 Canary routing: ~90% stable, ~10% canary")
print("=" * 55)
print()

counts = {}
N = 100
retries = 0
# Sleep between requests so we don't trip the 5r/s + burst=10 limiter. Note the
# sleep is OUTSIDE the retry branch: skipping the throttle on an error would
# make us hammer the gateway exactly when it is already rejecting us.
for _ in range(N):
    for attempt in range(4):
        r = requests.get(f"{GATEWAY}/api/canary/users")
        if r.status_code != 429:
            break
        retries += 1
        time.sleep(1)
    assert r.status_code == 200, f"gateway returned {r.status_code}: {r.text[:120]}"
    served = r.json()["served_by"]
    label = "stable (user-service-1)" if served.endswith("-1") else "canary (user-service-2)"
    counts[label] = counts.get(label, 0) + 1
    time.sleep(0.25)

print(f"Distribution across {N} requests:")
for label, count in sorted(counts.items()):
    bar = "█" * count
    pct = count / N * 100
    print(f"  {label:<30} {count:>3}  ({pct:4.1f}%)  {bar}")
if retries:
    print(f"  ({retries} request(s) were rate-limited and retried.)")
print()
print("\U0001f4a1 That is exactly 90/10, and it is supposed to be. nginx uses SMOOTH")
print("   weighted round-robin: a deterministic rotation with period sum(weights)=10")
print("   that spreads the canary request out rather than clumping it. Over 10 full")
print("   periods you get precisely 90 and 10 — no dice are rolled.")
print()
print("   Two things do make it drift in production:")
print("   • Multiple nginx workers. Each keeps its OWN rotation unless the upstream")
print("     declares a shared-memory `zone`. Our nginx.conf pins worker_processes 1")
print("     so this demo is reproducible; real configs use `zone` instead.")
print("   • Failed peers. A 502 consumes a slot and shifts the phase.")
print()
print("   ⚠️  Deterministic ≠ sticky. Consecutive requests from ONE user get")
print("      different versions, which breaks any flow with state. Use ip_hash or")
print("      a cookie-based split when a user must stay on one side.")

canary = counts.get("canary (user-service-2)", 0)
stable = counts.get("stable (user-service-1)", 0)

assert stable + canary == N, f"lost requests: {counts}"
# 9:1 smooth weighted round-robin over 10 whole periods is exact.
assert canary == N // 10 and stable == N - N // 10, (
    f"expected an exact 90/10 split from weights 9:1 over {N} requests, "
    f"got stable={stable} canary={canary} -- is worker_processes still 1?")
# And the split must be genuinely weighted, not the 50/50 of a plain upstream.
assert canary < stable / 4, (
    f"canary is taking {canary / N:.0%} of traffic -- the weights are not being applied")


### Production canary tips

- **Start tiny** (1–5%). Enough to see bugs, small enough that impact is limited.
- **Watch the right metrics** on the canary pool specifically: error rate, p95 latency, business KPIs (checkout success rate, etc.). Don't just watch CPU.
- **Automate the rollout.** Tools like **Argo Rollouts**, **Flagger**, and **AWS CodeDeploy** auto-promote or auto-rollback based on metrics.
- **Session stickiness matters** for stateful flows. If a user's first request hits the canary, you usually want follow-ups to also hit the canary (use `ip_hash` or a cookie-based router).
- **Feature flags are the other half.** Traffic-split canaries test infra; feature flags (LaunchDarkly, Unleash) test business behavior per user. Mature teams use both.

### Close cousins of canary routing

| Pattern | How it works | When to use |
|---------|------------|-------------|
| **Blue/Green** | Deploy v2 alongside v1, flip 100% of traffic at once | Fast rollback, but full blast radius on flip |
| **Canary** | Gradually shift 1% → 10% → 50% → 100% | Safer rollout, catches issues early |
| **Shadow / Mirror** | Duplicate traffic to v2 but ignore its responses | Validate performance with zero user risk |
| **A/B Test** | Route by user segment (country, cohort, flag) | Measure business impact of a change |


---

## 4️⃣ CORS (Cross-Origin Resource Sharing)

### The problem

Your web app is served from `https://app.example.com`. Its JavaScript tries to call `https://api.example.com/users`. By default, browsers **block** this — it's a "cross-origin" request. This is a security feature called the **Same-Origin Policy**.

To allow it, the API must respond with CORS headers:

```
Access-Control-Allow-Origin: https://app.example.com
Access-Control-Allow-Methods: GET, POST
Access-Control-Allow-Headers: Content-Type, X-API-Key
```

### Preflight requests

For any request that is **not** a "simple" GET/HEAD/POST-with-simple-headers, the browser first sends an `OPTIONS` request (called a **preflight**) to ask "may I?". If the preflight response has the right CORS headers, the browser sends the real request.

```
Browser              Gateway               Backend
───────              ───────               ───────
OPTIONS /api/...  ──▶                           (never reaches backend)
                     Access-Control-Allow-*
               ◀──── 204 No Content

GET /api/...      ──▶                     ──▶   real request
               ◀──── 200 + CORS headers   ◀──
```

### 🚫 BAD: Make every backend implement CORS

Duplicated headers in every service, inconsistent rules.

### 🏆 BEST: Handle CORS at the gateway

Our nginx config does this for `/api/cors/users`:

```nginx
location /api/cors/users {
    if ($request_method = OPTIONS) {
        add_header Access-Control-Allow-Origin  "*"                              always;
        add_header Access-Control-Allow-Methods "GET, POST, PUT, DELETE, OPTIONS" always;
        add_header Access-Control-Allow-Headers "Content-Type, X-API-Key"        always;
        return 204;
    }
    add_header Access-Control-Allow-Origin "*" always;
    proxy_pass http://user_backend/users;
}
```

Let's watch both halves of the exchange:


In [ ]:
print("🔎 CORS preflight (OPTIONS request)")
print("=" * 55)
r = requests.options(f"{GATEWAY}/api/cors/users",
    headers={
        "Origin": "https://app.example.com",
        "Access-Control-Request-Method": "GET",
        "Access-Control-Request-Headers": "X-API-Key",
    },
    timeout=3,
)
print(f"Status: {r.status_code}  (204 = OK, no body)")
for k, v in r.headers.items():
    if k.lower().startswith("access-control"):
        print(f"  {k}: {v}")

print()
print("🔎 Real request (GET)")
print("=" * 55)
r = requests.get(f"{GATEWAY}/api/cors/users",
                 headers={"Origin": "https://app.example.com"}, timeout=3)
print(f"Status: {r.status_code}")
for k, v in r.headers.items():
    if k.lower().startswith("access-control"):
        print(f"  {k}: {v}")


### Production CORS tips

- **Don't use `Access-Control-Allow-Origin: *` for credentialed requests.** If the browser sends cookies or auth headers, you must echo the exact origin back (and add `Access-Control-Allow-Credentials: true`).
- **Keep an allow-list of origins** instead of `*`. nginx `map` works well for this.
- **Set `Access-Control-Max-Age`** so browsers cache the preflight answer (reduces OPTIONS traffic).
- **Expose response headers** that JS needs to read (like `X-Request-ID`) via `Access-Control-Expose-Headers`.


---

## 5️⃣ Observability: Logging, Tracing, Monitoring

When a user says *"the app was slow yesterday"*, you need to reconstruct what happened across your services. You need **observability**:

- **Logs** — discrete events ("request X returned 500")
- **Metrics** — numbers over time ("p99 latency = 320ms")
- **Traces** — how a single request flowed through every service

The gateway is the perfect place to emit observability signals: **every** request passes through it.

### Structured access logs

Our `nginx.conf` writes JSON access logs to stdout:

```nginx
log_format gateway_json escape=json
    '{"time":"$time_iso8601",'
    '"request_id":"$request_id",'
    '"method":"$request_method",'
    '"uri":"$request_uri",'
    '"status":$status,'
    '"upstream_addr":"$upstream_addr",'
    '"upstream_response_time":"$upstream_response_time",'
    '"request_time":$request_time}';
access_log /dev/stdout gateway_json;
```

To watch the logs live while you use the notebook, open a second terminal:

```bash
docker logs -f api-gateway
```

### Distributed tracing with `X-Request-ID`

The gateway mints a unique ID for every request (nginx's built-in `$request_id`) and forwards it as `X-Request-ID`. If every service logs that ID, you can **grep across all logs** to reconstruct the journey of one request.


In [ ]:
import uuid

print("📝 Sending requests so the gateway writes access logs...")
correlation_tag = str(uuid.uuid4())[:8]  # our own tag (separate from X-Request-ID)

for i in range(5):
    r = requests.get(f"{GATEWAY}/api/users",
                     headers={"User-Agent": f"traffic-bot/{correlation_tag}"})
    print(f"  req {i+1}: status={r.status_code}")

print()
print("Run this in a terminal to see the structured JSON logs for our requests:")
print()
print(f"  docker logs api-gateway 2>&1 | grep {correlation_tag}")
print()
print("You'll see one JSON line per request with: request_id, status,")
print("upstream_addr (which backend served it), and request_time.")


In [ ]:
print("🧵 End-to-end trace ID propagation")
print("=" * 55)

for i in range(3):
    r = requests.get(f"{GATEWAY}/api/debug/headers")
    received = r.json()["received_headers"]
    gw_id = received.get("X-Request-Id")
    served_by = r.json()["served_by"]
    print(f"  req {i+1}: request_id={gw_id}  served_by={served_by}")

print()
print("💡 In production: log this ID in every service, then search by it in")
print("   Kibana/Datadog/Loki to see the entire trace in a single query.")
print()
print("🔗 Real tracing systems (OpenTelemetry, Jaeger, Zipkin) go further:")
print("   they correlate spans across services and show a visual timeline.")


### Metrics to collect at the gateway

If you add nothing else to your stack, these four gateway metrics give you 90% of the value:

1. **RPS per route** — how many requests per second per endpoint
2. **Error rate per route** — percentage of 4xx/5xx responses
3. **Latency per route** — p50, p95, p99 of `$request_time`
4. **Upstream health** — how often each backend instance failed (`$upstream_status`)

Tools that ingest nginx logs/metrics easily: **Prometheus** (via `nginx-prometheus-exporter`), **Grafana**, **Datadog**, **New Relic**, **CloudWatch**.


---

## 6️⃣ SSL / TLS Termination (concept-only in this lab)

Almost every production API is served over HTTPS. The gateway is where TLS is typically **terminated**: the client talks HTTPS to the gateway, and the gateway talks plain HTTP to internal services on a private network.

```
Client ──── HTTPS (TLS 1.3) ────▶ Gateway ──── HTTP (plain) ────▶ Backend
                                   │
                                   ├── Certificate management (cert renewal, rotation)
                                   ├── TLS version & cipher enforcement
                                   └── HSTS, HTTP → HTTPS redirect
```

### 🚫 BAD: Terminate TLS in every backend

Each service needs certs, cert rotation scripts, TLS library upgrades. Cert management becomes your #1 operational nightmare.

### 🏆 BEST: Terminate once at the gateway

The gateway owns certs. Backends stay simple. Example `nginx.conf` (this is in our file as a **commented reference** — the lab doesn't ship certs so it can't run):

```nginx
server {
    listen 443 ssl http2;
    server_name api.example.com;

    ssl_certificate     /etc/ssl/certs/api.example.com.pem;
    ssl_certificate_key /etc/ssl/private/api.example.com.key;

    ssl_protocols       TLSv1.2 TLSv1.3;
    ssl_ciphers         HIGH:!aNULL:!MD5;
    ssl_prefer_server_ciphers on;

    add_header Strict-Transport-Security "max-age=31536000; includeSubDomains" always;

    location /api/users {
        proxy_pass http://user_backend/users;   # plain HTTP internally
    }
}

# Redirect any accidental plain HTTP to HTTPS
server {
    listen 80;
    server_name api.example.com;
    return 301 https://$host$request_uri;
}
```

### Where to get certs

- **Let's Encrypt** — free, automated, 90-day certs (use `certbot`)
- **AWS ACM / GCP / Azure** — free when used with their load balancers
- **Your internal CA** — for private APIs

### mTLS — when the gateway also verifies the client

Standard TLS verifies the **server**. In high-security setups (service-to-service traffic, B2B APIs), **mutual TLS (mTLS)** also verifies the **client's certificate**. nginx supports this via `ssl_client_certificate` + `ssl_verify_client on`. This is common inside service meshes (Istio enables it automatically between services).


---

## 🧰 Real-World API Gateway Products

We used **nginx OSS** in this lab because it's free, ubiquitous, and teaches you the primitives. Production teams pick from a wider menu, and the right choice depends on *what layer* the gateway lives at and *how much logic* you want to push into it.

| Product | Type | Strengths | When to pick it |
|---------|------|-----------|-----------------|
| **nginx / nginx Plus** | Reverse proxy + gateway | Fast, simple, config-as-file | Small teams, self-hosted, want full control |
| **Envoy** | L7 proxy (C++) | True circuit breakers, retries, gRPC, observability built in | Base of service meshes; powers Istio, Consul, AWS App Mesh |
| **Kong** | API gateway (nginx + Lua plugins) | Rich plugin ecosystem (auth, transforms, quotas), admin API | Self-hosted platforms that want extensibility without writing Lua |
| **Traefik** | Cloud-native reverse proxy | Auto-discovery from Docker/Kubernetes, Let's Encrypt built in | Kubernetes-first teams, small ops footprint |
| **HAProxy** | L4/L7 load balancer | Extremely fast, rock-solid, great TCP support | Very high-RPS edges, non-HTTP protocols |
| **AWS API Gateway** | Fully managed | Pay-per-request, deep AWS integration (Lambda, IAM, Cognito, WAF) | Serverless/AWS-heavy stacks; don't want to run servers |
| **Azure API Management (APIM)** | Fully managed | Developer portal, policies-as-XML, Azure AD integration | Azure-heavy enterprises, API productization |
| **Google Cloud API Gateway / Apigee** | Managed | Apigee adds analytics, monetization, versioning UI | GCP stacks; large enterprise API programs |
| **Cloudflare / Fastly** | Edge gateway (CDN + gateway) | Global POPs, DDoS protection, WAF, Workers/compute at edge | Public APIs that need global low-latency + protection |
| **Istio / Linkerd** (Ingress Gateway) | Service mesh edge | mTLS, fine-grained traffic policies, canary + retries | Kubernetes with many services talking to each other |

### Quick heuristic

- **Managed (AWS API Gateway / Azure APIM / Apigee)** if you want *less ops* and your stack already lives there.
- **Kong** if you want a self-hosted gateway with a plugin ecosystem.
- **Envoy / Istio** if you need production-grade resilience (true circuit breakers, smart retries, mTLS) and you're on Kubernetes.
- **nginx / HAProxy / Traefik** if you want a battle-tested L7 proxy with a small footprint and no vendor lock-in.
- **Cloudflare / Fastly** if you want the gateway at the *edge*, closer to users, with DDoS and WAF included.

The patterns you learned in this lab (routing, rate limiting, auth, retries, canary, CORS, TLS termination, observability) translate 1:1 to every product above. The *config syntax* differs; the *concepts* don't.



---

## 📚 Summary

### What We Learned

| Pattern | Core idea | nginx primitive | Production upgrade |
|---------|----------|-----------------|---------------------|
| Circuit-breaker-style failover | Don't send traffic to a dead instance | `max_fails`, `fail_timeout`, `proxy_next_upstream` | Envoy / Istio / Resilience4j |
| Retry amplification | Nested retries multiply, worst exactly when the backend is sick | `proxy_next_upstream_tries` (kept at 2) | Retry budgets + backoff with jitter |
| Timeout budgets | Each hop must allow strictly less time than the one above | `proxy_read_timeout` > service-level timeout | Deadline propagation (gRPC deadlines, context) |
| Request aggregation (BFF) | One client call = many backend calls | URL route → service that composes | Dedicated BFF / GraphQL gateway |
| Canary / weighted routing | Ship a new version to a small slice of users | `server ... weight=N;` | Argo Rollouts / Flagger / service mesh |
| CORS | Let other-origin browsers call your API | `add_header Access-Control-*` | Same, plus origin allow-list |
| Observability | Logs + traces correlated by request ID | `log_format` + `$request_id` | OpenTelemetry / Jaeger / Datadog |
| TLS termination | Encrypt at the edge, plain internally | `listen 443 ssl; ssl_certificate ...` | Managed certs (ACM / Let's Encrypt) |

### Key Takeaways

1. Basic gateways (nginx OSS) handle *most* production needs: routing, LB, rate limiting, auth, CORS, TLS, basic failover. Reach for a service mesh when you need true circuit breakers, fine-grained retries, or mTLS by default.
2. A circuit breaker's value is the calls it **doesn't make**. In the simulation above it spared the backend 6 of 15 requests while it was down — retrying into a dead backend is just load.
3. **Retries multiply, they don't add.** The amplification factor is the product of the per-layer retry counts, and it peaks exactly when the backend can least afford it. Retry at one layer, with a budget and jittered backoff, and never on non-idempotent operations.
4. **Timeout budgets must shrink with depth.** The headroom at each level is what buys you a degraded-but-useful answer instead of a dead connection. And a retry count without a total time cap (`proxy_next_upstream_timeout`) is not a bounded budget.
5. Aggregation is a real win on mobile — but mostly in connections, payload size and partial-failure handling, not raw latency (a client can parallelise its own calls). It also moves the fan-out into the BFF, which now owns the timeouts and the blast radius.
6. **Never report an unverified fact as a number.** When the orders call fails, `order_count` is `null`, not `0` — `0` is a claim you didn't check.
7. Observability is worthless without **correlation**: a `request_id` logged in every hop is the single most useful thing you can add.
8. TLS belongs at the edge. Backends shouldn't know about certificates.

### Complete Gateway Capability Matrix (all 4 notebooks)

| Notebook | Capability |
|----------|-----------|
| 1 | Path-based routing, load balancing, health checks |
| 2 | Rate limiting, API key authentication |
| 3 | Header injection, API versioning, URL rewriting |
| 4 | Circuit-breaker-style failover, retry amplification, timeout budgets, request aggregation (BFF) with partial-failure handling, canary routing, CORS, observability, TLS termination |

That's the complete picture of what an API gateway does in production. 🎉
